# Using the New Rashomon-Set API

This tutorial is a small, self-contained demo of the redesigned Rashomon-set
search in `src.interval_utils.compute_rashomon_set`. The new API gives you
three things the old one didn't:

1. **Choose the certification method.** The optimization loop that grows the
   parameter box always uses fast IBP, but the *reported* certificates can be
   computed with any registered verification method (`IBP`, `CROWN`,
   `alpha-CROWN`, ...) via `certification_method`.
2. **Certify against input regions, not just points.** Pass
   `has_input_intervals=True` and a dataset of `(x_l, x_u, y)` triples to
   certify the Rashomon set against state *regions* (e.g. sensor noise),
   instead of single observations.
3. **Specify a single number: the minimum accuracy you want.** A
   `target_accuracy` used to come bundled with four other easy-to-misalign
   parameters (`soft_min`/`hard_min`/`soft_metric`/`soft_temperature`/
   `aggregation`) inside an `AccuracyRequirement` wrapper class; now `accuracy`
   is just a plain `float` (or `dict[group_id, float]`) passed directly - no
   wrapper needed. From that one number,
   `compute_rashomon_set` builds the differentiable surrogate (a softmax
   margin), its dataset-level aggregation (an exact order statistic - not
   `"mean"` or `"min"`), and calibrates the softmax temperature it depends
   on, all automatically. An explicit `group_by` function (replacing the old
   `task_labels` list) still lets you give different groups different
   targets.

We'll train a tiny toy classifier and walk through each of these in turn.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import torch.nn as nn


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start
    for candidate in (start, *start.parents):
        if (candidate / "core").is_dir() and (candidate / "tutorials").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current directory.")


REPO_ROOT = find_repo_root()
CORE_PATH = REPO_ROOT / "core"
if str(CORE_PATH) not in sys.path:
    sys.path.insert(0, str(CORE_PATH))

from src.interval_utils import compute_rashomon_set  # noqa: E402
from src.rashomon_spec import RashomonResult, resolve_accuracy  # noqa: E402

SEED = 0
torch.manual_seed(SEED)
REPO_ROOT


PosixPath('/vol/bitbucket/ma5923/_projects/CertifiedContinualLearning')

## 1. A Tiny Toy Classifier

A 2-class dataset, linearly separable by `x0 + x1 - x2 > 0`, and a small
`Linear -> ReLU -> Linear` network trained to (near-)perfectly classify it.
The Rashomon set is the set of weight perturbations around this trained
network that still meet an accuracy requirement we'll specify below.

In [2]:
N = 200
X = torch.randn(N, 3)
y = ((X[:, 0] + X[:, 1] - X[:, 2]) > 0).long()

model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()
for _ in range(300):
    optimizer.zero_grad()
    loss = loss_fn(model(X), y)
    loss.backward()
    optimizer.step()
model.eval()

with torch.no_grad():
    train_acc = (model(X).argmax(dim=1) == y).float().mean().item()
print(f"final loss: {loss.item():.4f}, train accuracy: {train_acc:.4f}")


def summarize(result: RashomonResult, label: str) -> None:
    """Pretty-print the last checkpoint's certificates and calibrated temperatures."""
    print(f"{label}: {len(result.bounded_models)} checkpoint(s) returned")
    for cert in result.certificates[-1]:
        group_str = f"group={cert.group}" if cert.group is not None else "global"
        tau = result.temperatures[cert.group]
        print(
            f"  {group_str:<10} surrogate>={cert.min_surrogate:.3f}  "
            f"hard_acc>={cert.min_hard_acc:.3f}  tau={tau:.3f}"
        )


final loss: 0.0224, train accuracy: 0.9950


## 2. `accuracy`: One Number Replaces Five

The minimum certified accuracy used to be specified via `AccuracyRequirement`,
a wrapper bundling five separate, easy-to-misalign parameters (`soft_min` /
`hard_min` / `soft_metric` / `soft_temperature` / `aggregation`). Now `accuracy`
is just a plain value:

- `accuracy`: the minimum fraction of certificate samples that must
  be correctly (and certifiably) classified. A single float shared by every
  group, or a `dict[group_id, float]` for per-group targets.

From `accuracy` alone, `compute_rashomon_set` constructs the rest
automatically:

- **Pointwise surrogate**: a softmax margin (probability mass on the
  admissible/correct action(s), minus a soundness threshold) - what used to
  be the `"accuracy_margin"` `soft_metric` choice is now the only surrogate.
- **Dataset-level aggregation**: not `"mean"` or `"min"`, but the *exact*
  `(1 - accuracy)`-quantile order statistic of the per-sample
  margins. `accuracy=1.0` recovers the old strict `"min"` (every
  sample must pass); lower values tolerate a precise fraction of failures -
  see section 4 below.
- **Softmax temperature**: calibrated automatically against the nominal
  (pre-optimization) model rather than supplied by hand - see section 7.

`resolve_accuracy(accuracy, group)` looks up the target accuracy for a given
group id - this is exactly what `compute_rashomon_set` calls internally to
pick the order-statistic rank.

In [3]:
shared = 0.9
print("shared target, any group:", resolve_accuracy(shared, 0), resolve_accuracy(shared, 1))

per_group = {0: 0.5, 1: 0.95}
print("per-group target, group 0:", resolve_accuracy(per_group, 0))
print("per-group target, group 1:", resolve_accuracy(per_group, 1))


shared target, any group: 0.9 0.9
per-group target, group 0: 0.5
per-group target, group 1: 0.95


## 3. Computing a Rashomon Set

By default, `compute_rashomon_set` treats the whole dataset as a single
global group: it grows a box around the trained weights as large as possible
while keeping the certified accuracy above `target_accuracy` (via the
auto-built margin surrogate and order-statistic aggregation described
above). The optimization loop is capped at a small `n_iters` here so the
tutorial runs quickly - in practice you'd use the (much larger) default of
2000.

Calibration searches from sharp to flat (doubling from `tau=0.1` up to
`tau=100.0`) and stops at the *first* (sharpest) candidate that already
clears the order-statistic margin. For a well-fit model the margin clears
immediately, so calibration usually lands on the smallest candidate
temperature (`0.1`). A sharp softmax gives the Lagrangian loop a strong,
well-behaved gradient signal for the accuracy constraint - this is the fix
for a once-real box-collapse failure mode, where a flat calibrated
temperature gave a too-weak constraint gradient relative to the
(always-strong) box-growing objective. Calibration only flattens (raises
`tau`) as much as necessary to clear the margin, because a low `tau` also
raises the order statistic's false-negative rate - a single hard-failing
sample's margin is most catastrophic at low `tau`, so it's not flattened
further than it needs to be. Even with a sharp, well-behaved gradient, the
box-growing objective is still much stronger than an already-easy
constraint, so `primal_learning_rate=5e-4` below is still used to keep the
two forces balanced for this toy problem. If you see the soft accuracy
collapse to exactly `-0.5` (the worst possible margin), an imbalance
between these two forces - not a bug in the surrogate itself - is almost
always the cause.

In [4]:
# compute_rashomon_set requires y pre-encoded as a (N, n_classes) multi-hot tensor
# (single-label targets are the M=1 special case: one-hot).
dataset = torch.utils.data.TensorDataset(X, torch.nn.functional.one_hot(y, num_classes=2).float())
PRIMAL_LR = 5e-4  # see note above: keeps box growth from outrunning a low calibrated temperature

result = compute_rashomon_set(
    model, dataset,
    0.85,
    batch_size=N, certificate_samples=N, n_iters=150, primal_learning_rate=PRIMAL_LR,
)
summarize(result, "single global group")


Calibrated temperatures: {None: 0.1}
Initial acc constraint violation (group=None): -0.5000 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with target accuracies: {None: 0.85}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

...

 92%|█████████▏| 138/150 [00:01<00:00, 110.66it/s, size=2.54, obj=0.050, min_surrogate=0.500]

 92%|█████████▏| 138/150 [00:01<00:00, 110.66it/s, size=2.56, obj=0.051, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 110.02it/s, size=2.56, obj=0.051, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 111.32it/s, size=2.56, obj=0.051, min_surrogate=0.500]

Final bbox:  Obj=0.05,  Size=2.56,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]
Computing final certificates over 200 samples using IBP
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
single global group: 1 checkpoint(s) returned
  global     surrogate>=0.500  hard_acc>=1.000  tau=0.100


## 4. `target_accuracy` Is an Exact Tolerance, Not a Vague Average

`target_accuracy=1.0` means literally every certificate sample must pass -
the order-statistic rank picked is the strict minimum, identical to the old
`aggregation="min"`. Lower values tolerate a precise number of failures: with
`N` certificate samples, `target_accuracy=p` allows `floor((1-p)*N)` of them
to fail. Asking for `1.0` on noisy real data is often infeasible (or forces
a tiny box); relaxing it slightly tends to grow the box substantially for a
small, well-understood drop in certified coverage.

In [5]:
for target_accuracy in (1.0, 0.85, 0.6):
    try:
        result_strictness = compute_rashomon_set(
            model, dataset,
            target_accuracy,
            batch_size=N, certificate_samples=N, n_iters=150, primal_learning_rate=PRIMAL_LR,
        )
        summarize(result_strictness, f"target_accuracy={target_accuracy}")
    except ValueError as exc:
        # target_accuracy=1.0 demands literally every one of the N=200 certificate
        # samples pass - with a 99.5%-accurate model that's almost always infeasible,
        # and no amount of calibration can rescue a genuinely misclassified sample.
        print(f"target_accuracy={target_accuracy}: calibration failed - {exc}")


  group=None: target_accuracy=1.000, achieved hard accuracy=0.995  [INFEASIBLE]
target_accuracy=1.0: 1 checkpoint(s) returned
  global     surrogate>=0.000  hard_acc>=0.995  tau=0.000
Calibrated temperatures: {None: 0.1}
Initial acc constraint violation (group=None): -0.5000 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with target accuracies: {None: 0.85}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

...

 91%|█████████▏| 137/150 [00:01<00:00, 122.72it/s, size=2.54, obj=0.050, min_surrogate=0.500]

 91%|█████████▏| 137/150 [00:01<00:00, 122.72it/s, size=2.56, obj=0.051, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 117.26it/s, size=2.56, obj=0.051, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 99.74it/s, size=2.56, obj=0.051, min_surrogate=0.500] 

Final bbox:  Obj=0.05,  Size=2.56,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]
Computing final certificates over 200 samples using IBP
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
target_accuracy=0.6: 1 checkpoint(s) returned
  global     surrogate>=0.500  hard_acc>=1.000  tau=0.100


## 5. Per-Group Accuracy Requirements via `group_by`

`group_by` replaces the old `task_labels` list. It's a function applied to
each minibatch's `y` (the (N, n_classes) multi-hot tensor) that returns an
integer group id per sample - each unique id gets its own Lagrangian
constraint with its own `resolve_accuracy(accuracy, group)` target (and its
own independently-calibrated temperature - see section 8). For single-label
classification, `group_by=lambda y: y.argmax(dim=-1)` recovers "one
constraint per class", but `group_by` works just as well on genuinely
multi-hot admissible-action-set targets (anything that can be turned into an
integer group id).

Here we ask for a *looser* requirement on class 0 and a *stricter* one on
class 1, simulating a setting where one class is more safety-critical than
the other.

In [6]:
result_grouped = compute_rashomon_set(
    model, dataset,
    {0: 0.6, 1: 0.95},
    batch_size=N, certificate_samples=N, n_iters=150, primal_learning_rate=PRIMAL_LR / 2,
    group_by=lambda y: y.argmax(dim=-1),
)
summarize(result_grouped, "per-class group_by")


Calibrated temperatures: {0: 0.1, 1: 0.1}
Initial acc constraint violation (group=0): -0.5000 (Positive = violated)
Initial acc constraint violation (group=1): -0.5000 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with target accuracies: {0: 0.6, 1: 0.95}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=0, min_surrogate=0.5, min_hard_acc=1.0), RashomonCertificate(group=1, min_surrogate=0.5, min_hard_acc=1.0)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

...

 93%|█████████▎| 140/150 [00:01<00:00, 96.55it/s, size=1.27, obj=0.025, min_surrogate=0.500]

 93%|█████████▎| 140/150 [00:01<00:00, 96.55it/s, size=1.28, obj=0.025, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 96.35it/s, size=1.28, obj=0.025, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 95.63it/s, size=1.28, obj=0.025, min_surrogate=0.500]

Final bbox:  Obj=0.03,  Size=1.28,  Certificates=[RashomonCertificate(group=0, min_surrogate=0.5, min_hard_acc=1.0), RashomonCertificate(group=1, min_surrogate=0.4999973773956299, min_hard_acc=1.0)]
Computing final certificates over 200 samples using IBP
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
per-class group_by: 1 checkpoint(s) returned
  group=0    surrogate>=0.500  hard_acc>=1.000  tau=0.100
  group=1    surrogate>=0.500  hard_acc>=1.000  tau=0.100


## 6. Certifying Against Input Regions, Not Just Points

Pass `has_input_intervals=True` together with a dataset that yields
`(x_l, x_u, y)` triples (instead of `(x, y)`) to certify the Rashomon set
against an input *region* around each sample - e.g. sensor noise or
discretization error - rather than the single observed point. Internally,
every `bound_forward` call inside the optimization and certification now
uses `(x_l, x_u)` instead of `(x, x)`.

Widening the input region can only make the certified accuracy go down or
stay the same (the model has to be correct on harder, perturbed inputs too),
so we expect a lower certificate than the point-only run above for the same
accuracy target.

In [7]:
eps = 0.02
X_l, X_u = X - eps, X + eps
interval_dataset = torch.utils.data.TensorDataset(
    X_l, X_u, torch.nn.functional.one_hot(y, num_classes=2).float()
)

result_interval = compute_rashomon_set(
    model, interval_dataset,
    0.85,
    batch_size=N, certificate_samples=N, n_iters=150, primal_learning_rate=PRIMAL_LR,
    has_input_intervals=True,
)
summarize(result_interval, f"certified against a +-{eps} input box")


Calibrated temperatures: {None: 0.1}
Initial acc constraint violation (group=None): -0.5000 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with target accuracies: {None: 0.85}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

...

 95%|█████████▍| 142/150 [00:01<00:00, 106.09it/s, size=2.52, obj=0.050, min_surrogate=0.500]

 95%|█████████▍| 142/150 [00:01<00:00, 106.09it/s, size=2.54, obj=0.050, min_surrogate=0.500]

 95%|█████████▍| 142/150 [00:01<00:00, 106.09it/s, size=2.56, obj=0.051, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 115.17it/s, size=2.56, obj=0.051, min_surrogate=0.500]

Final bbox:  Obj=0.05,  Size=2.56,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]
Computing final certificates over 200 samples using IBP
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
certified against a +-0.02 input box: 1 checkpoint(s) returned
  global     surrogate>=0.500  hard_acc>=1.000  tau=0.100


## 7. Choosing the Certification Method

The Lagrangian loop that grows the parameter box always uses `IBP` - it's
fast, and the loop runs hundreds to thousands of iterations. But the
*reported* certificates don't have to use the same method: pass
`certification_method` (and optionally `certification_method_kwargs`) to
rebuild a `BoundedModel` from each checkpoint's `(param_l, param_u)` using
any registered verification method, and certify against that instead. This
reuses `build_bounded_model` from `src.verification.api` under the hood.

Different methods make different soundness/tightness trade-offs, so the
reported certificate can differ between methods even though the optimized
box itself is identical.

In [8]:
for method in ["IBP", "CROWN"]:
    result_method = compute_rashomon_set(
        model, dataset,
        0.85,
        batch_size=N, certificate_samples=N, n_iters=150, primal_learning_rate=PRIMAL_LR,
        certification_method=method,
    )
    summarize(result_method, method)


Calibrated temperatures: {None: 0.1}
Initial acc constraint violation (group=None): -0.5000 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with target accuracies: {None: 0.85}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

...

 95%|█████████▍| 142/150 [00:01<00:00, 112.62it/s, size=2.52, obj=0.050, min_surrogate=0.500]

 95%|█████████▍| 142/150 [00:01<00:00, 112.62it/s, size=2.54, obj=0.050, min_surrogate=0.500]

 95%|█████████▍| 142/150 [00:01<00:00, 112.62it/s, size=2.56, obj=0.051, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 111.24it/s, size=2.56, obj=0.051, min_surrogate=0.500]

Final bbox:  Obj=0.05,  Size=2.56,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]
Computing final certificates over 200 samples using CROWN
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
CROWN: 1 checkpoint(s) returned
  global     surrogate>=0.500  hard_acc>=1.000  tau=0.100


## 8. Reusing a Calibrated Temperature Across Calls

Calibration happens once per call, against the nominal (pre-optimization)
model, and independently per group. Sometimes several related
`compute_rashomon_set` calls need to use the *same* temperature - e.g.
comparing multiple reference models against one fixed surrogate, rather than
letting each call recalibrate (and potentially land on a different
temperature). Pass `temperatures` explicitly to skip calibration entirely
and use exactly the value(s) given - `RashomonResult.temperatures` from an
earlier call is exactly the right shape to feed back in.

In [9]:
calibrated = result.temperatures
print("Auto-calibrated temperature from section 3:", calibrated)

result_frozen = compute_rashomon_set(
    model, dataset,
    0.85,
    batch_size=N, certificate_samples=N, n_iters=150, primal_learning_rate=PRIMAL_LR,
    temperatures=calibrated,
)
summarize(result_frozen, "reusing the calibrated temperature")
assert result_frozen.temperatures == calibrated


Auto-calibrated temperature from section 3: {None: 0.1}
Calibrated temperatures: {None: 0.1}
Initial acc constraint violation (group=None): -0.5000 (Positive = violated)
Number of model parameters: 50
Computing Rashomon set with target accuracies: {None: 0.85}
Initial bbox:  Obj=0.00,  Size=0.00,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]


  0%|          | 0/150 [00:00<?, ?it/s]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

  0%|          | 0/150 [00:00<?, ?it/s, size=0.00, obj=0.000, min_surrogate=0.500]

...

 96%|█████████▌| 144/150 [00:01<00:00, 113.54it/s, size=2.52, obj=0.050, min_surrogate=0.500]

 96%|█████████▌| 144/150 [00:01<00:00, 113.54it/s, size=2.54, obj=0.050, min_surrogate=0.500]

 96%|█████████▌| 144/150 [00:01<00:00, 113.54it/s, size=2.56, obj=0.051, min_surrogate=0.500]

100%|██████████| 150/150 [00:01<00:00, 113.24it/s, size=2.56, obj=0.051, min_surrogate=0.500]

Final bbox:  Obj=0.05,  Size=2.56,  Certificates=[RashomonCertificate(group=None, min_surrogate=0.5, min_hard_acc=1.0)]
Computing final certificates over 200 samples using IBP
Num cert samples: 200
----------------------- Finished Computing Rashomon set ------------------------
reusing the calibrated temperature: 1 checkpoint(s) returned
  global     surrogate>=0.500  hard_acc>=1.000  tau=0.100


## Notes

- `IntervalTrainer.compute_rashomon_set` (in `src.trainer.IntervalTrainer`)
  is a thin wrapper around this same function for continual-learning
  pipelines: it resolves a trainer-level accuracy target (a plain float or
  per-group dict, with its own `min_acc_increment` policy) and forwards
  `group_by`, `has_input_intervals`, `certification_method`,
  `certification_method_kwargs`, and `temperatures` straight through,
  exposing the result's calibrated temperatures as `self.temperatures`.
- `RashomonResult.certificates` is always a list (one entry per checkpoint)
  of lists (one `RashomonCertificate` per group) - a uniform shape regardless
  of whether `group_by` was given. `RashomonResult.temperatures` is a single
  dict (one entry per group) shared across all checkpoints, since
  calibration happens once, up front, against the nominal model.
- `checkpoint=N` in `compute_rashomon_set` records a deep copy of the
  optimization-time box every `N` iterations (plus the final one); each
  checkpoint gets its own certificates, which is useful for tracking how the
  Rashomon set's certified accuracy evolves as the box grows.
- Margin soundness holds for *any* softmax temperature `tau>0` - calibration
  only reduces false negatives (samples that are actually certifiable but
  whose margin hasn't separated from 0 yet at a given `tau`), it never
  affects correctness. Lower `tau` is preferred when it's enough to clear
  the margin (sharper, better-behaved gradients for the optimizer), so
  calibration searches from sharp to flat and stops at the first (smallest)
  `tau` that works. If no `tau` in `[0.1, 100.0]` clears the order-statistic
  margin, `compute_rashomon_set` raises a `ValueError` rather than silently
  using a miscalibrated surrogate.